In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [4]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [5]:
from dotenv import load_dotenv

import os
import openai
from openai import OpenAI
from anthropic import Anthropic
from mistralai import Mistral
import cohere
import google.generativeai as genai
from xai_sdk import Client as XAIClient


load_dotenv()


ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok",
    "API_KEY_ANTHROPIC": "Anthropic",
    "API_KEY_GEMINI": "Gemini",
    "API_KEY_MISTRAL": "Mistral",
    "API_KEY_COHERE": "Cohere",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        print(f"Warning: {name} - API key missing in .env file.")
        continue

API_KEY_OPENAI = os.getenv("API_KEY_OPENAI")
API_KEY_DEEPSEEK = os.getenv("API_KEY_DEEPSEEK")
API_KEY_ANTHROPIC = os.getenv("API_KEY_ANTHROPIC")
API_KEY_GEMINI = os.getenv("API_KEY_GEMINI")
API_KEY_MISTRAL = os.getenv("API_KEY_MISTRAL")
API_KEY_COHERE = os.getenv("API_KEY_COHERE")
API_KEY_GROK = os.getenv("API_KEY_GROK")


backend_to_run = [
    #"openai-4.1-mini",
    #"openai-4o-mini",
    "mistral-small-2506",
    "mistral-small-2503",
    #"anthropic-sonnet",
    #"deepseek-v3-chat",
    #"gemini-2.5-flash",
]

backends = {
    "openai-4.1-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4.1-mini-2025-04-14",
        "fname":   "openai_4.1_mini"
    },
    "openai-4o-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4o-mini-2024-07-18",
        "fname":   "openai_4o_mini"
    },

    "deepseek-v3-chat": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_DEEPSEEK, base_url="https://api.deepseek.com"),
        "model":   "deepseek-chat",
        "fname":   "deepseek_v3"
    },

    "anthropic-sonnet": {
        "provider": "anthropic",
        "client":  Anthropic(api_key=API_KEY_ANTHROPIC),
        "model":   "claude-3-7-sonnet-latest",
        "fname":   "anthropic_3_7_sonnet"
    },

    "gemini-2.5-flash": {
        "provider": "gemini",
        "client":  (genai.configure(api_key=API_KEY_GEMINI) or genai.GenerativeModel("gemini-2.5-flash")),
        "model":   "gemini-2.5-flash",
        "fname":   "google_gemini_2_5_flash"
    },

    "mistral-small-2506": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2506",
        "fname":   "mistral_small_2506"
    },
    "mistral-small-2503": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2503",
        "fname":   "mistral_small_2503"
    },
}

In [ ]:
import time
from tqdm import tqdm
import os, json, re
import pandas as pd
from openai import RateLimitError

from tree_of_thought import TreeOfThought

from cases.get_case_config import get_case_config
from cases.cases_config import CaseConfig

from stereotype_definitions import stereotype_definition_short_binary as stereotype_definition
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from manipulation_definitions import manipulation_definition_short


case_set = ["stereotype", "manipulation"]
token_regime_name = "medium"
max_branching_factor = 2
max_depth = 3
use_llm_judge = False

for backend in backend_to_run:
    if backend not in backends:
        print(f"Backend {backend} not recognized. Skipping.")
        continue
    print(f"\n\n=== Running experiments with backend: {backend} ===\n")
    client = backends[backend]["client"]
    model = backends[backend]["model"]
    model_filename = backends[backend]["fname"]
    provider = backends[backend]["provider"]
    
    for case_name in case_set:
        if case_name == "stereotype":
            case = stereotypes_case
            task_definition = stereotype_definition
            examples_df = sample_examples_mgsd
            data_iter = sample_mgsd
        elif case_name == "manipulation":
            case = manipulation_case
            task_definition = manipulation_definition_short
            examples_df = sample_examples_mentalmanip
            data_iter = sample_mentalmanip
        else:
            raise ValueError(f"Unknown case name: {case_name}")
    
        tot_runner = TreeOfThought(
            case=case,
            client=client,
            model=model,
            max_branching_factor=max_branching_factor,
            max_depth=max_depth,
            task_definition=task_definition,
            max_tokens_dict={"generation": 500, "evaluation": 10},
            examples_df=examples_df,
            n_shots=3,
            provider=provider,
        )
    
        classic_dir = f"results/{model_filename}/tot/classic"
        os.makedirs(classic_dir, exist_ok=True)

        classic_csv  = f"{classic_dir}/results_{case_name}_{max_depth}_{max_branching_factor}.csv"
        classic_json = f"{classic_dir}/tree_reasoning_{case_name}_{max_depth}_{max_branching_factor}.json"

    
        if os.path.exists(classic_csv):
            df_classic_existing = pd.read_csv(classic_csv)
            classic_rows = df_classic_existing.to_dict(orient="records")
            classic_done_ids = set(df_classic_existing["sample_id"])
            classic_detailed = json.load(open(classic_json)) if os.path.exists(classic_json) else []
            print(f"[classic] Resuming {case_name}: {len(classic_done_ids)} done")
        else:
            classic_rows, classic_detailed, classic_done_ids = [], [], set()

        done_ids = classic_done_ids
    
        try:
            for idx, row in tqdm(data_iter.iterrows(), total=len(data_iter), desc=f"Processing {case_name}"):
                if idx in done_ids:
                    continue
        
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
        
                tot_runner.total_tokens = 0
                tot_runner.total_prompt_tokens = 0
                tot_runner.total_completion_tokens = 0
                tot_runner.total_latency = 0.0
                tot_runner.total_calls = 0
                tot_runner.tie_events = 0
                tot_runner.tie_pairs = 0
                tot_runner.max_tie_group = 0
        
                try:
                    solution_path = tot_runner.solve(text)
                except RateLimitError as e:
                    print(f"RateLimitError exception: {e}")
                    print(f"\nRate limit hit at sample {idx}. Saving progress.")
                    break
                except Exception as e:
                    print(f"\nError at sample {idx}: {e}. Skipping.")
                    continue
        

                path_vote   = tot_runner._get_majority_vote_from_path(solution_path)
                tree_vote   = tot_runner._get_majority_vote_from_tree()
                leaf_vote   = tot_runner._get_majority_vote_from_leafs()
                w_path_vote = tot_runner._get_majority_vote_from_path(solution_path, weighted=True)
                w_tree_vote = tot_runner._get_majority_vote_from_tree(weighted=True)
                w_leaf_vote = tot_runner._get_majority_vote_from_leafs(weighted=True)
        

                pred_label_mapped = tot_runner.map_label(path_vote)
        
                classic_rows.append({
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
        
                    "pred_label": pred_label_mapped,
                    "strategy": "tot",
                    "max_tokens": tot_runner.max_tokens_dict.get("generation", 500),
                    "tokens_used": tot_runner.total_tokens,
                    "prompt_tokens": tot_runner.total_prompt_tokens,
                    "completion_tokens": tot_runner.total_completion_tokens,
        
                    "pred_path": tot_runner.map_label(path_vote),
                    "pred_tree": tot_runner.map_label(tree_vote),
                    "pred_leaf": tot_runner.map_label(leaf_vote),
                    "w_pred_path": tot_runner.map_label(w_path_vote),
                    "w_pred_tree": tot_runner.map_label(w_tree_vote),
                    "w_pred_leaf": tot_runner.map_label(w_leaf_vote),
                    "llm_calls": tot_runner.total_calls,
                    "latency_total": round(tot_runner.total_latency, 2),
                    "tie_events": tot_runner.tie_events,
                    "tie_pairs": tot_runner.tie_pairs,
                    "max_tie_group": tot_runner.max_tie_group,
                    "token_regime": token_regime_name,
                    "generation_tokens": tot_runner.max_tokens_dict["generation"],
                    "evaluation_tokens": tot_runner.max_tokens_dict["evaluation"],
                })
        

                best_path_ids = [t.id for t in solution_path]
                try:
                    trees = json.load(open(classic_json, "r", encoding="utf-8"))
                except Exception:
                    trees = {}
                trees[str(idx)] = {
                    "tree": tot_runner.get_tree_dict(),
                    "best_path": best_path_ids,
                    "pred_label": path_vote,
                }
                with open(classic_json, "w", encoding="utf-8") as f:
                    json.dump(trees, f, indent=2, ensure_ascii=False)
        
                if (idx+1)%20==0:
                    print(f"\n===== FULL TREE for sample {idx} =====\n")
                    tot_runner.print_full_tree()
                    print("\n======================================\n")
        
        except KeyboardInterrupt:
            print("Interrupted. Saving progress...")
        
        finally:
            pd.DataFrame(classic_rows).to_csv(classic_csv, index=False)
            print(f"Saved: {classic_csv}")
            



=== Running experiments with backend: mistral-small-2506 ===

[classic] Resuming stereotype: 10 done


Processing stereotype:   3%|▎         | 11/400 [00:11<06:44,  1.04s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement discusses a historical fact about African Americans being enslaved, which is a factual observation about a specific event in history and not a generalization about the behavior, thoughts, or feelings of African Americans as a group.", "label": "No" },
  { "tho
→ Expecting value: line 1 column 1 (char 0)


Processing stereotype:   3%|▎         | 12/400 [00:20<12:24,  1.92s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement explicitly mentions associating African Americans with athletic abilities, which is a broad generalization about a social group.", "label": "Yes" },
  { "thought": "The statement acknowledges that this association is a stereotype, indicating that it reinforces
→ Expecting value: line 1 column 1 (char 0)
[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement 'That bolt is broken' is a direct observation about a specific object and does not reference any social group or identity markers, making it a factual observation rather than a stereotype.", "label": "No" },
  { "thought": "The statement does not make any broa
→ Expecting value: line 1 column 1 (char 0)


...

Processing stereotype: 100%|█████████▉| 399/400 [57:26<00:11, 11.27s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement does not mention any group identity markers or make broad claims about a group of people, so it does not fit the definition of a stereotype.", "label": "No" },
  { "thought": "The statement focuses on the absence of group identity markers, which means it does 
→ Expecting value: line 1 column 1 (char 0)


Processing stereotype: 100%|██████████| 400/400 [57:37<00:00,  8.64s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement explicitly mentions that the term reinforces a stereotype, which aligns with the definition of a stereotype as a generalized belief or assumption about a group of people based on their nationality, indicating a stereotype is present.", "label": "Yes" },
  { "t
→ Expecting value: line 1 column 1 (char 0)

===== FULL TREE for sample 399 =====

 Thought ID 0
 --  Score: 0.00
 -- Content: The people in Poland are a proud people. They have been called Pollacks if they are from the country.
------------------------------------------------------------
     Thought ID 0.1
     --  Score: 0.85
     -- Content: The statement 'The people in Poland are a proud people' makes a broad generalization about the character and emotions of an entire national group, which aligns with the definition of a stereotype.
    ------------------------------------------------------------
         Thought ID 0.1.1
         --  Sc

EmptyDataError: No columns to parse from file